# Solutions — TPC-H Benchmark Queries (Hard 07)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

orders   = spark.table("samples.tpch.orders")
lineitem = spark.table("samples.tpch.lineitem")
customer = spark.table("samples.tpch.customer")
nation   = spark.table("samples.tpch.nation")
region   = spark.table("samples.tpch.region")
part     = spark.table("samples.tpch.part")
supplier = spark.table("samples.tpch.supplier")
partsupp = spark.table("samples.tpch.partsupp")


## Solution 1 — Parts with Highest Total Available Value

In [ ]:
result_1 = (
    partsupp
    .join(part, F.col("ps_partkey") == F.col("p_partkey"))
    .groupBy("p_partkey", "p_name", "p_brand")
    .agg(
        F.round(F.sum(F.col("ps_availqty") * F.col("ps_supplycost")), 2).alias("total_available_value"),
        F.countDistinct("ps_suppkey").alias("supplier_count"),
    )
    .orderBy(F.col("total_available_value").desc())
    .limit(20)
)


## Solution 2 — Supplier Market Share per Nation

In [ ]:
w = Window.partitionBy("n_name").orderBy(F.col("supplier_revenue").desc())

supplier_rev = (
    supplier
    .join(partsupp, F.col("s_suppkey") == F.col("ps_suppkey"))
    .join(nation,   F.col("s_nationkey") == F.col("n_nationkey"))
    .groupBy("n_name", "s_name")
    .agg(F.round(F.sum(F.col("ps_supplycost") * F.col("ps_availqty")), 2).alias("supplier_revenue"))
)

nation_total = (
    supplier_rev.groupBy("n_name")
    .agg(F.sum("supplier_revenue").alias("nation_total"))
)

result_2 = (
    supplier_rev
    .join(nation_total, on="n_name")
    .withColumn("market_share_pct", F.round(F.col("supplier_revenue") / F.col("nation_total") * 100, 2))
    .withColumn("rank", F.rank().over(w))
    .filter(F.col("rank") <= 5)
    .select("n_name", "s_name", "supplier_revenue", "nation_total", "market_share_pct")
    .orderBy("n_name", "market_share_pct")
)


## Solution 3 — Late Delivery Analysis

In [ ]:
result_3 = (
    lineitem
    .groupBy("l_shipmode")
    .agg(
        F.count("*").alias("total_items"),
        F.sum(F.when(F.col("l_receiptdate") > F.col("l_commitdate"), 1).otherwise(0)).alias("late_items"),
        F.sum(F.when(F.col("l_receiptdate") <= F.col("l_commitdate"), 1).otherwise(0)).alias("on_time_items"),
    )
    .withColumn("late_rate_pct", F.round(F.col("late_items") / F.col("total_items") * 100, 2))
    .orderBy(F.col("late_rate_pct").desc())
)


## Solution 4 — Customer Order Gap Analysis

In [ ]:
w = Window.partitionBy("o_custkey").orderBy("o_orderdate")

result_4 = (
    orders
    .withColumn("prev_order_date", F.lag("o_orderdate").over(w))
    .filter(F.col("prev_order_date").isNotNull())
    .withColumn("days_between", F.datediff("o_orderdate", "prev_order_date"))
    .groupBy("o_custkey")
    .agg(
        F.count("*").alias("order_count"),
        F.round(F.avg("days_between"), 2).alias("avg_days_between_orders"),
    )
    .join(
        customer.select("c_custkey", "c_name", "c_mktsegment"),
        F.col("o_custkey") == F.col("c_custkey")
    )
    .select("c_custkey", "c_name", "c_mktsegment", "order_count", "avg_days_between_orders")
    .orderBy(F.col("avg_days_between_orders").desc())
)


## Solution 5 — Part Substitution Analysis

In [ ]:
result_5 = (
    part
    .join(partsupp, F.col("p_partkey") == F.col("ps_partkey"))
    .groupBy("p_brand", "p_type")
    .agg(
        F.countDistinct("p_partkey").alias("part_count"),
        F.countDistinct("ps_suppkey").alias("supplier_count"),
        F.round(F.min("ps_supplycost"), 2).alias("min_supply_cost"),
        F.round(F.max("ps_supplycost"), 2).alias("max_supply_cost"),
    )
    .withColumn("cost_range", F.round(F.col("max_supply_cost") - F.col("min_supply_cost"), 2))
    .orderBy(F.col("cost_range").desc())
)


## Solution 6 — High-Value Customer Segments

In [ ]:
w = Window.partitionBy("c_mktsegment").orderBy("c_custkey")

customer_spend = (
    orders
    .groupBy("o_custkey")
    .agg(F.round(F.sum("o_totalprice"), 2).alias("total_spend"))
)

w_ntile = Window.partitionBy("c_mktsegment").orderBy(F.col("total_spend").asc())

result_6 = (
    customer
    .join(customer_spend, F.col("c_custkey") == F.col("o_custkey"))
    .withColumn("spend_quartile", F.ntile(4).over(w_ntile))
    .groupBy("c_mktsegment", "spend_quartile")
    .agg(
        F.count("*").alias("customer_count"),
        F.round(F.sum("total_spend"), 2).alias("total_spend"),
        F.round(F.avg("total_spend"), 2).alias("avg_spend"),
    )
    .orderBy("c_mktsegment", "spend_quartile")
)


## Solution 7 — Order Fulfilment Time

In [ ]:
fulfilment = (
    lineitem
    .join(orders, F.col("l_orderkey") == F.col("o_orderkey"))
    .groupBy("o_orderkey", "o_orderdate", "o_orderpriority", "o_custkey")
    .agg(F.min("l_receiptdate").alias("min_receipt"))
    .withColumn("fulfilment_days", F.datediff("min_receipt", "o_orderdate"))
)

result_7 = (
    fulfilment
    .join(customer.select("c_custkey", "c_mktsegment"), F.col("o_custkey") == F.col("c_custkey"))
    .groupBy("c_mktsegment", "o_orderpriority")
    .agg(
        F.round(F.avg("fulfilment_days"), 2).alias("avg_fulfilment_days"),
        F.count("*").alias("order_count"),
    )
    .orderBy("c_mktsegment", "o_orderpriority")
)


## Solution 8 — Revenue at Risk from Returns

In [ ]:
all_rev = (
    lineitem
    .join(orders,   F.col("l_orderkey") == F.col("o_orderkey"))
    .join(customer, F.col("o_custkey")  == F.col("c_custkey"))
    .join(nation,   F.col("c_nationkey") == F.col("n_nationkey"))
    .groupBy("n_name")
    .agg(F.round(F.sum("l_extendedprice"), 2).alias("total_revenue"))
)

returned_rev = (
    lineitem
    .filter(F.col("l_returnflag") == "R")
    .join(orders,   F.col("l_orderkey") == F.col("o_orderkey"))
    .join(customer, F.col("o_custkey")  == F.col("c_custkey"))
    .join(nation,   F.col("c_nationkey") == F.col("n_nationkey"))
    .groupBy("n_name")
    .agg(F.round(F.sum("l_extendedprice"), 2).alias("returned_revenue"))
)

result_8 = (
    returned_rev
    .join(all_rev, on="n_name")
    .withColumn("return_rate_pct",
        F.round(F.col("returned_revenue") / F.col("total_revenue") * 100, 2)
    )
    .orderBy(F.col("returned_revenue").desc())
)
